# Fine-Tuning Qwen3.5-0.8B dengan Unsloth (Google Colab)

> **JALANKAN NOTEBOOK INI PALING DULU.** Model terkecil & tercepat -> dipakai untuk
> validasi seluruh pipeline (data load -> train -> eval -> merge -> save) sebelum
> menghabiskan kuota GPU untuk Llama-3.2-1B (01) dan Gemma3-1B (02).

Notebook ini full Unsloth + Colab (GPU T4/L4). Dataset dan `max_seq_length` diambil dari
`Data/processed_shared/` yang dihasilkan oleh `00_analisis_dataset.ipynb` -> jalankan itu
terlebih dahulu dan upload seluruh folder project ke
`Google Drive/Fine-Tune SLM for Medical Chatbot/` (struktur sama persis: `Data/`, `notebooks/`, `outputs/`).

**Prasyarat:**
- Runtime -> Change runtime type -> **GPU (T4 atau lebih baik)**
- `Data/processed_shared/training_config_recommended.json` sudah ada di Drive

**Config training** (dinaikkan dari versi non-Unsloth karena Unsloth jauh lebih hemat VRAM):
- `BATCH_SIZE=4`, `GRAD_ACCUM=4` (effective batch 16, lebih cepat per step)
- `EPOCHS=3` (Unsloth ~2x lebih cepat, headroom untuk epoch tambahan)

> Jika OOM: turunkan `BATCH_SIZE` ke 2 dan naikkan `GRAD_ACCUM` ke 8.

> Kuota terbatas: training otomatis **resume dari checkpoint terakhir** jika session
> Colab terputus -> tinggal jalankan ulang cell training.

In [ ]:
%%capture
!pip install unsloth
%pip install -q evaluate rouge_score matplotlib

In [ ]:
import os, gc, json
import torch
from pathlib import Path

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

assert torch.cuda.is_available(), 'CUDA tidak tersedia! Set Runtime -> Change runtime type -> GPU.'

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')

print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'bf16   : {torch.cuda.is_bf16_supported()}')
print(f'PyTorch: {torch.__version__}')

## Konfigurasi

In [ ]:
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/Fine-Tune SLM for Medical Chatbot'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path(DRIVE_PROJECT_DIR)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    BASE_DIR = Path(os.getcwd()).parent
    if not (BASE_DIR / 'Data').exists():
        BASE_DIR = Path(os.getcwd())

assert (BASE_DIR / 'Data').exists(), f'Data/ tidak ditemukan di {BASE_DIR}'

SHARED_DIR = BASE_DIR / 'Data' / 'processed_shared'
DATA_DIR   = BASE_DIR / 'Data' / 'processed'
OUT_DIR    = BASE_DIR / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# HF_TOKEN opsional - varian unsloth/* sudah public/ungated, tapi tetap dibaca kalau ada
# (mengurangi rate-limit HF dan dipakai kalau PUSH_TO_HUB=True)
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN and IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN') or ''
    except Exception:
        pass

MODEL_ID    = 'unsloth/Qwen3.5-0.8B'
ADAPTER_DIR = str(OUT_DIR / 'checkpoints' / 'qwen35-0.8b-medical-adapter')
MERGED_DIR  = str(OUT_DIR / 'merged'      / 'qwen35-0.8b-medical')
CKPT_DIR    = str(OUT_DIR / 'checkpoints' / 'qwen35-0.8b-train')

TRAIN_RED_FILE = str(SHARED_DIR / 'train_reduced.jsonl')
VAL_RED_FILE   = str(SHARED_DIR / 'val_reduced.jsonl')
TEST_FILE      = str(DATA_DIR   / 'test.jsonl')

with open(str(SHARED_DIR / 'training_config_recommended.json')) as _f:
    _cfg = json.load(_f)
MAX_SEQ_LEN = _cfg['max_seq_length']
print(f'MAX_SEQ_LEN  : {MAX_SEQ_LEN}')
print(f'Target train : {_cfg["target_train_samples"]:,}')

LORA_R      = 16
LORA_ALPHA  = 32
LORA_DROP   = 0.05
LORA_TARGET = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
               'gate_proj', 'up_proj', 'down_proj']

# Config dinaikkan dari versi non-Unsloth (BATCH_SIZE 2->4, GRAD_ACCUM 8->4, EPOCHS 2->3)
# karena Unsloth jauh lebih hemat VRAM & lebih cepat per step.
BATCH_SIZE    = 4
GRAD_ACCUM    = 4
LEARNING_RATE = 2e-4
EPOCHS        = 3
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.05
SEED          = 42

DO_MERGE    = True
PUSH_TO_HUB = False
HF_REPO     = 'ariesdjae/qwen35-0.8b-medical'

print(f'Mode     : {"Colab (Drive)" if IN_COLAB else "Lokal"}')
print(f'Model    : {MODEL_ID}')
print(f'EffBatch : {BATCH_SIZE * GRAD_ACCUM}')
print(f'\nPath check:')
print(f'  SHARED_DIR     : {SHARED_DIR}  {"OK" if SHARED_DIR.exists() else "TIDAK ADA"}')
print(f'  DATA_DIR       : {DATA_DIR}  {"OK" if DATA_DIR.exists() else "TIDAK ADA"}')
print(f'  train_reduced  : {"OK" if Path(TRAIN_RED_FILE).exists() else "TIDAK ADA"}')
print(f'  val_reduced    : {"OK" if Path(VAL_RED_FILE).exists() else "TIDAK ADA"}')
print(f'  test.jsonl     : {"OK" if Path(TEST_FILE).exists() else "TIDAK ADA"}')

In [ ]:
from datasets import load_dataset

train_ds = load_dataset('json', data_files=TRAIN_RED_FILE, split='train')
val_ds   = load_dataset('json', data_files=VAL_RED_FILE,   split='train')
print(f'Train : {len(train_ds):,}  Val : {len(val_ds):,}')

In [ ]:
from unsloth import FastLanguageModel

print(f'Loading {MODEL_ID} ...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,            # auto-detect (bf16 di T4/L4/A100)
    load_in_4bit=True,
    token=HF_TOKEN or None,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

torch.cuda.empty_cache(); gc.collect()
total = sum(p.numel() for p in model.parameters())
print(f'Params : {total / 1e9:.2f}B')
print(f'VRAM   : {torch.cuda.memory_allocated() / 1e9:.2f} GB')

In [ ]:
def format_chat(sample):
    # Qwen3.5 mungkin thinking model -> enable_thinking=False agar output hanya jawaban
    try:
        text = tokenizer.apply_chat_template(
            sample['messages'], tokenize=False,
            add_generation_prompt=False, enable_thinking=False
        )
    except TypeError:
        text = tokenizer.apply_chat_template(
            sample['messages'], tokenize=False, add_generation_prompt=False
        )
    return {'text': text}

# Colab (Linux, fork) -> num_proc>1 aman untuk closure yang mereferensikan tokenizer
train_ds = train_ds.map(format_chat, remove_columns=train_ds.column_names, num_proc=2)
val_ds   = val_ds.map(format_chat,   remove_columns=val_ds.column_names,   num_proc=2)
print(train_ds[0]['text'][:400])

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROP,
    bias='none',
    use_gradient_checkpointing='unsloth',  # Unsloth: checkpointing lebih hemat VRAM
    random_state=SEED,
)
model.print_trainable_parameters()

In [ ]:
from trl import SFTConfig, SFTTrainer
import glob

os.makedirs(CKPT_DIR, exist_ok=True)
USE_BF16 = torch.cuda.is_bf16_supported()

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        output_dir=CKPT_DIR,
        dataset_text_field='text',
        max_length=MAX_SEQ_LEN,
        packing=True,
        dataset_num_proc=2,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_ratio=WARMUP_RATIO,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        optim='adamw_8bit',
        lr_scheduler_type='cosine',
        fp16=not USE_BF16,
        bf16=USE_BF16,
        max_grad_norm=1.0,
        logging_steps=50,
        eval_strategy='epoch',
        save_strategy='best',          # exempt dari syarat eval==save strategy + load_best_model_at_end
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        report_to='none',
        dataloader_num_workers=0,
        dataloader_pin_memory=False,
        seed=SEED,
    ),
)

# Kuota terbatas -> tahan disconnect: lanjut dari checkpoint terakhir jika ada
_ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, 'checkpoint-*')),
                 key=lambda p: int(p.rsplit('-', 1)[-1]))
_resume = _ckpts[-1] if _ckpts else None
if _resume:
    print(f'Resuming from checkpoint: {_resume}')

torch.cuda.empty_cache()
result = trainer.train(resume_from_checkpoint=_resume)
print(f'Train loss : {result.training_loss:.4f}  Steps : {result.global_step}')

## Kurva Learning Rate & Loss

In [ ]:
import matplotlib.pyplot as plt

log_hist   = trainer.state.log_history
lr_steps   = [(h['step'], h['learning_rate']) for h in log_hist if 'learning_rate' in h]
loss_steps = [(h['step'], h['loss'])          for h in log_hist if 'loss' in h]
eval_steps = [(h['step'], h['eval_loss'])     for h in log_hist if 'eval_loss' in h]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

if lr_steps:
    xs, ys = zip(*lr_steps)
    ax[0].plot(xs, ys)
ax[0].set_title('Learning Rate Schedule')
ax[0].set_xlabel('step'); ax[0].set_ylabel('learning rate')

if loss_steps:
    xs, ys = zip(*loss_steps)
    ax[1].plot(xs, ys, label='train_loss')
if eval_steps:
    xs, ys = zip(*eval_steps)
    ax[1].plot(xs, ys, 'o-', label='eval_loss')
ax[1].set_title('Loss')
ax[1].set_xlabel('step'); ax[1].set_ylabel('loss'); ax[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(CKPT_DIR, 'training_curves.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
os.makedirs(ADAPTER_DIR, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'Adapter saved -> {ADAPTER_DIR}')

## Evaluasi ROUGE-L

In [ ]:
if not os.path.exists(TEST_FILE):
    print(f'[SKIP] test.jsonl tidak ditemukan: {TEST_FILE}')
else:
    from datasets import load_dataset as _lds
    from evaluate import load as load_metric

    rouge  = load_metric('rouge')
    SYSTEM = ('You are a helpful medical assistant. '
              'Answer questions accurately based on clinical guidelines and ICD-11 classification.')

    FastLanguageModel.for_inference(model)  # Unsloth: 2x lebih cepat untuk inferensi
    torch.cuda.empty_cache()

    preds, refs = [], []
    test_ds = _lds('json', data_files=TEST_FILE, split='train')
    n_eval  = min(200, len(test_ds))
    for sample in test_ds.select(range(n_eval)):
        msgs = sample['messages']
        ref  = next(m['content'] for m in msgs if m['role'] == 'assistant')
        prompt_msgs = [m for m in msgs if m['role'] != 'assistant']
        try:
            text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False,
                                                  add_generation_prompt=True,
                                                  enable_thinking=False)
        except TypeError:
            text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False,
                                                  add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors='pt', truncation=True,
                           max_length=MAX_SEQ_LEN - 256).to('cuda')
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
        preds.append(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],
                                       skip_special_tokens=True))
        refs.append(ref)
        del inputs, out
        torch.cuda.empty_cache()

    scores = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
    for k, v in scores.items():
        print(f'  {k}: {v:.4f}')

## Uji inferensi kualitatif

In [ ]:
SYSTEM = ('You are a helpful medical assistant. '
          'Answer questions accurately based on clinical guidelines and ICD-11 classification.')
test_questions = [
    'Apa gejala utama demam berdarah dengue dan kapan harus ke rumah sakit?',
    'Bagaimana cara menangani pasien dengan nyeri dada akut?',
    'What are the first-line treatment options for Type 2 Diabetes Mellitus?',
    'Apakah antibiotik diperlukan untuk infeksi virus seperti flu biasa?',
    'Dok saya hamil 8 minggu dan mengalami mual parah, apa yang harus saya lakukan?',
]
FastLanguageModel.for_inference(model)  # Unsloth: 2x lebih cepat untuk inferensi
for q in test_questions:
    messages = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': q}]
    try:
        text = tokenizer.apply_chat_template(messages, tokenize=False,
                                              add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to('cuda')
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=350, do_sample=True,
                             temperature=0.7, top_p=0.9, repetition_penalty=1.1,
                             pad_token_id=tokenizer.eos_token_id)
    resp = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'Q: {q}\nA: {resp[:400]}\n' + '-' * 70)
    del inputs, out
    torch.cuda.empty_cache()

## Merge adapter -> model penuh

In [ ]:
if DO_MERGE:
    os.makedirs(MERGED_DIR, exist_ok=True)
    model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')
    print(f'Merged saved -> {MERGED_DIR}')
else:
    print('DO_MERGE=False. Adapter tersimpan di', ADAPTER_DIR)

In [ ]:
if PUSH_TO_HUB:
    if not HF_TOKEN:
        raise ValueError('Set HF_TOKEN di Colab Secrets untuk push ke HuggingFace Hub')
    model.push_to_hub_merged(HF_REPO, tokenizer, save_method='merged_16bit', token=HF_TOKEN)
    print(f'Model pushed -> huggingface.co/{HF_REPO}')
else:
    print(f'Output lokal:\n  Adapter: {ADAPTER_DIR}')
    if DO_MERGE:
        print(f'  Merged : {MERGED_DIR}')